# 03 — Pipeline modular: agents, questionários e análise

Este notebook usa as funções encapsuladas em módulos para rodar o pipeline.

Diferença principal em relação ao notebook anterior:

- `REPEATS = 1` por padrão.
- Todos os 11 constructos psicográficos entram no questionário padrão.
- As funções ficam separadas por responsabilidade:
  - agentes;
  - questionários;
  - artefatos;
  - análise/agregação.


In [2]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm")

# Ajuste este caminho caso copie a pasta synthetic_audiences para src/
MODULE_DIR = PROJECT_DIR
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

OUTPUT_DIR = PROJECT_DIR / "data" / "output"
INPUT_DIR = PROJECT_DIR / "data" / "input"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_DIR)


C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm


## 1. Imports dos módulos

In [ ]:
from src.synthetic_audiences.questionnaires.constructs import (
    build_stimulus_response_questionnaire, aggregate_construct_scores
)
from src.synthetic_audiences.artifacts.news import build_news_artifact

from src.synthetic_audiences.questionnaires.runner import run_questionnaire

from src.synthetic_audiences.analysis.aggregation import (
    aggregate_by_construct,
    aggregate_by_variables_construct,
    aggregate_by_agent_construct,
    build_pivot_segment_construct,
    
)


## 2. Ler a base de agentes

In [3]:
AGENTS_CSV = OUTPUT_DIR / "agents_population_n1000_seed42.csv"

if not AGENTS_CSV.exists():
    candidate_files = sorted(OUTPUT_DIR.glob("agents_population*.csv"))
    if candidate_files:
        AGENTS_CSV = candidate_files[0]

df_agents = pd.read_csv(AGENTS_CSV)

print("Arquivo:", AGENTS_CSV)
print("Agentes:", len(df_agents))
df_agents.head(3)


Arquivo: C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm\data\output\agents_population_n1000_seed42.csv
Agentes: 1000


,id_persona,idade,gênero,escolaridade,situação ocupacional,estado civil,condição de moradia,tipo de moradia,localização,renda familiar mensal,...,Meritocracia,Ideologia,Discriminação,Sexualidade,Comportamento_social,Identidade_nacional,Insegurança,Confiança,Persona,renda_pc
0,agent_2cc5d8ce623f,50-64 anos,Feminino,Ensino médio completo,Empregado com carteira assinada,Solteiro(a),Imóvel alugado,Apartamento,Capital,Mais de 20 salários mínimos,...,"Expressa crença em esforço individual, merecim...","Usar identidade política, conservadorismo/prog...","Mede reconhecimento de racismo, machismo, prec...","Captura aceitação de diversidade sexual, toler...","Traduz sociabilidade cotidiana, hábitos de míd...","Mede orgulho/vergonha do Brasil, povo brasilei...","Captura medo da violência, vulnerabilidade per...","Mede confiança interpessoal, família, vizinhos...","Pessoa de 50-64 anos, Feminino, Parda, residen...",Mais de 1 SM
1,agent_29ef94c4ffbf,35-49 anos,Feminino,Ensino técnico / profissionalizante,Funcionário público,Casado(a),Imóvel cedido por familiar ou terceiros,Casa,Capital,Mais de 5 a 10 salários mínimos,...,"Expressa crença em esforço individual, merecim...","Usar identidade política, conservadorismo/prog...","Mede reconhecimento de racismo, machismo, prec...","Captura aceitação de diversidade sexual, toler...","Traduz sociabilidade cotidiana, hábitos de míd...","Mede orgulho/vergonha do Brasil, povo brasilei...","Captura medo da violência, vulnerabilidade per...","Mede confiança interpessoal, família, vizinhos...","Pessoa de 35-49 anos, Feminino, Branca, reside...",Mais de 1 SM
2,agent_ac2595547fe5,50-64 anos,Feminino,Ensino técnico / profissionalizante,Empregador / empresário,Casado(a),Imóvel próprio quitado,Apartamento,Capital,Mais de 3 a 5 salários mínimos,...,"Expressa crença em esforço individual, merecim...","Usar identidade política, conservadorismo/prog...","Mede reconhecimento de racismo, machismo, prec...","Captura aceitação de diversidade sexual, toler...","Traduz sociabilidade cotidiana, hábitos de míd...","Mede orgulho/vergonha do Brasil, povo brasilei...","Captura medo da violência, vulnerabilidade per...","Mede confiança interpessoal, família, vizinhos...","Pessoa de 50-64 anos, Feminino, Branca, reside...",Mais de 1 SM


## 3. Criar o questionário

Por padrão, este notebook usa os 11 constructos psicográficos:

1. Religiosidade
2. Família
3. Honestidade
4. Meritocracia
5. Ideologia
6. Discriminação
7. Sexualidade
8. Comportamento social
9. Identidade nacional
10. Insegurança
11. Confiança

Também deixei um questionário complementar de reação à notícia, caso você queira medir confiança institucional, risco, compartilhamento e valência.


In [ ]:


# Use apenas os 11 constructos:
questionnaire = build_stimulus_response_questionnaire()
# Ou, se quiser juntar os dois:
# questionnaire = construct_questionnaire + media_questionnaire

print("Total de perguntas:", len(questionnaire))
pd.DataFrame(questionnaire)[["question_id", "construct", "question"]]


Total de perguntas: 11


,question_id,construct,question
0,C001,Religiosidade,"Considerando esta notícia, quanto sua interpre..."
1,C002,Família,Quanto esta notícia se conecta com sua visão s...
2,C003,Honestidade,"Quanto esta notícia parece envolver confiança,..."
3,C004,Meritocracia,Quanto esta notícia se conecta com sua visão s...
4,C005,Ideologia,Quanto sua posição política ou ideológica infl...
5,C006,Discriminação,Quanto esta notícia faz você pensar em desigua...
6,C007,Sexualidade,Quanto esta notícia se conecta com sua visão s...
7,C008,Comportamento_social,Quanto esta notícia se conecta com seus hábito...
8,C009,Identidade_nacional,"Quanto esta notícia influencia seu orgulho, ve..."
9,C010,Insegurança,Quanto esta notícia aumenta sua sensação de me...


## 4. Criar artefato de notícia

In [5]:
artifact = build_news_artifact(
    artifact_id="NEWS_001",
    artifact_type="news_text",
    title="Empresa anuncia investimento em tecnologia e sustentabilidade",
    tipo_de_fontes_de_informação="Alta confiança histórica e institucional: jornais de referência",
    source_name="Jornal de referência",
    content="""
Uma grande empresa brasileira anunciou um novo plano de investimento em tecnologia,
inteligência artificial e sustentabilidade. Segundo a companhia, a iniciativa deve aumentar
a eficiência operacional, reduzir emissões e abrir novas oportunidades de emprego qualificado.

Especialistas avaliam que o plano pode fortalecer a imagem institucional da empresa,
mas organizações da sociedade civil cobram mais transparência sobre metas ambientais,
governança dos dados e impactos sobre trabalhadores em funções operacionais.
""".strip(),
)

artifact


{'artifact_id': 'NEWS_001',
 'artifact_type': 'news_text',
 'title': 'Empresa anuncia investimento em tecnologia e sustentabilidade',
 'content': 'Uma grande empresa brasileira anunciou um novo plano de investimento em tecnologia,\ninteligência artificial e sustentabilidade. Segundo a companhia, a iniciativa deve aumentar\na eficiência operacional, reduzir emissões e abrir novas oportunidades de emprego qualificado.\n\nEspecialistas avaliam que o plano pode fortalecer a imagem institucional da empresa,\nmas organizações da sociedade civil cobram mais transparência sobre metas ambientais,\ngovernança dos dados e impactos sobre trabalhadores em funções operacionais.',
 'tipo_de_fontes_de_informação': 'Alta confiança histórica e institucional: jornais de referência',
 'source_name': 'Jornal de referência'}

## 5. Rodar questionário com REPEATS=1

In [6]:
LIMIT_AGENTS = 10
REPEATS = 1

df_responses, metadata = run_questionnaire(
    df_agents=df_agents,
    questionnaire=questionnaire,
    artifact=artifact,
    output_dir=OUTPUT_DIR,
    limit_agents=LIMIT_AGENTS,
    repeats=REPEATS,
    sleep_seconds=0.2,
)

metadata


{'run_id': 'run_langchain_20260616_151533_bea791',
 'responses_jsonl': 'C:\\Users\\claudio.bruno_cortex\\Documents\\código_python\\synthetic-audiences-abm\\data\\output\\run_langchain_20260616_151533_bea791_responses.jsonl',
 'responses_csv': 'C:\\Users\\claudio.bruno_cortex\\Documents\\código_python\\synthetic-audiences-abm\\data\\output\\run_langchain_20260616_151533_bea791_responses.csv',
 'errors': [],
 'n_responses': 110,
 'n_errors': 0}

In [7]:
df_responses.head()


,run_id,model,artifact_id,artifact_type,artifact_title,artifact_tipo_de_fontes_de_informação,id_persona,Persona,idade,gênero,...,answer_value,answer_label,rationale,confidence,fonte_familiar,source_familiarity_effect,persona_summary,socio_demographic_summary,values_attitudes_perceptions_summary,raw_answer
0,run_langchain_20260616_151533_bea791,gpt-4o-mini,NEWS_001,news_text,Empresa anuncia investimento em tecnologia e s...,Alta confiança histórica e institucional: jorn...,agent_2cc5d8ce623f,"Pessoa de 50-64 anos, Feminino, Parda, residen...",50-64 anos,Feminino,...,2,Discordo parcialmente / baixo,"Embora eu valorize a diversidade e a ética, mi...",0.7,True,A familiaridade com a fonte me deixou mais rec...,"Como uma mulher progressista, minha visão é mo...",Minha renda e ocupação me permitem uma perspec...,"Valorizo a ética e a justiça social, mas minha...","{""answer_value"": 2, ""answer_label"": ""Discordo ..."
1,run_langchain_20260616_151533_bea791,gpt-4o-mini,NEWS_001,news_text,Empresa anuncia investimento em tecnologia e s...,Alta confiança histórica e institucional: jorn...,agent_2cc5d8ce623f,"Pessoa de 50-64 anos, Feminino, Parda, residen...",50-64 anos,Feminino,...,4,Concordo parcialmente / alto,A notícia sobre investimento em tecnologia e s...,0.8,True,A familiaridade com a fonte me deixou mais rec...,"Como uma mulher da Geração Bossa Nova, valoriz...",Minha condição de renda alta e moradia na capi...,Minha orientação progressista me leva a valori...,"{""answer_value"": 4, ""answer_label"": ""Concordo ..."
2,run_langchain_20260616_151533_bea791,gpt-4o-mini,NEWS_001,news_text,Empresa anuncia investimento em tecnologia e s...,Alta confiança histórica e institucional: jorn...,agent_2cc5d8ce623f,"Pessoa de 50-64 anos, Feminino, Parda, residen...",50-64 anos,Feminino,...,4,Concordo parcialmente / alto,A notícia apresenta um investimento em tecnolo...,0.8,True,A familiaridade com a fonte me faz interpretar...,"Como uma pessoa da Geração Bossa Nova, valoriz...",Minha renda e educação me permitem exigir mais...,Minha orientação progressista me leva a questi...,"{""answer_value"": 4, ""answer_label"": ""Concordo ..."
3,run_langchain_20260616_151533_bea791,gpt-4o-mini,NEWS_001,news_text,Empresa anuncia investimento em tecnologia e s...,Alta confiança histórica e institucional: jorn...,agent_2cc5d8ce623f,"Pessoa de 50-64 anos, Feminino, Parda, residen...",50-64 anos,Feminino,...,4,Concordo parcialmente / alto,Acredito que investimentos em tecnologia e sus...,0.8,True,A familiaridade com a fonte me faz ver a notíc...,"Como uma mulher progressista, valorizo iniciat...",Minha situação de renda e emprego me faz valor...,Meus valores progressistas me levam a apoiar o...,"{""answer_value"": 4, ""answer_label"": ""Concordo ..."
4,run_langchain_20260616_151533_bea791,gpt-4o-mini,NEWS_001,news_text,Empresa anuncia investimento em tecnologia e s...,Alta confiança histórica e institucional: jorn...,agent_2cc5d8ce623f,"Pessoa de 50-64 anos, Feminino, Parda, residen...",50-64 anos,Feminino,...,4,Concordo parcialmente / alto,Minha posição política progressista me faz val...,0.8,True,A familiaridade com a fonte me deixou mais rec...,"Como uma mulher progressista, valorizo iniciat...",Minha renda alta e formação me permitem ter um...,Meus valores progressistas influenciam minha i...,"{""answer_value"": 4, ""answer_label"": ""Concordo ..."


## 6. Análises e agregações

In [ ]:
df_construct_scores = aggregate_construct_scores(df_responses)

In [8]:
run_id = metadata["run_id"]

df_construct = aggregate_by_construct(
    df_responses,
    output_path=OUTPUT_DIR / f"{run_id}_aggregate_by_construct.csv",
)

df_variables_construct = aggregate_by_variables_construct(
    df_responses,
    output_path=OUTPUT_DIR / f"{run_id}_aggregate_by_variables_construct.csv",
)

df_agent_construct = aggregate_by_agent_construct(
    df_responses,
    output_path=OUTPUT_DIR / f"{run_id}_aggregate_by_agent_construct.csv",
)

pivot_segment = build_pivot_segment_construct(
    df_variables_construct,
    output_path=OUTPUT_DIR / f"{run_id}_pivot_segment_construct.csv",
)

df_construct


,artifact_id,construct,mean_answer,std_answer,median_answer,n_responses,n_agents,mean_confidence
0,NEWS_001,Comportamento_social,3.0,1.054,3.0,10,10,0.75
1,NEWS_001,Confiança,3.0,1.054,3.0,10,10,0.75
2,NEWS_001,Discriminação,3.2,0.789,3.0,10,10,0.72
3,NEWS_001,Família,3.4,0.699,3.5,10,10,0.71
4,NEWS_001,Honestidade,3.0,1.054,3.0,10,10,0.75
5,NEWS_001,Identidade_nacional,3.4,0.699,3.5,10,10,0.74
6,NEWS_001,Ideologia,4.1,0.316,4.0,10,10,0.78
7,NEWS_001,Insegurança,2.6,0.699,2.5,10,10,0.73
8,NEWS_001,Meritocracia,3.8,0.422,4.0,10,10,0.73
9,NEWS_001,Religiosidade,2.5,0.850,2.0,10,10,0.74


In [9]:
pivot_segment


construct,Comportamento_social,Confiança,Discriminação,Família,Honestidade,Identidade_nacional,Ideologia,Insegurança,Meritocracia,Religiosidade,Sexualidade
group_value,,,,,,,,,,,
Agro,2.0,2.0,3.0,3.0,2.0,3.0,4.0,3.0,4.0,3.0,2.0
Conservadores cristãos,2.0,2.0,2.5,3.0,2.0,3.0,4.0,3.0,4.0,4.0,2.0
Dependentes do Estado / Classe D e E,2.0,2.0,4.0,2.5,2.0,2.5,4.0,3.5,3.0,2.0,2.0
Empresários,4.0,4.0,2.0,4.0,4.0,4.0,4.0,2.0,4.0,2.0,2.0
Liberais sociais,4.0,4.0,3.0,4.0,4.0,4.0,4.0,2.0,4.0,2.0,3.0
Progressistas,4.0,4.0,4.0,4.0,4.0,4.0,4.5,2.0,4.0,2.0,3.5


## 7. Auditoria qualitativa

In [10]:
audit_cols = [
    "id_persona",
    "segmento_e_bolhas",
    "geração",
    "ideologia",
    "construct",
    "answer_value",
    "answer_label",
    "fonte_familiar",
    "rationale",
    "source_familiarity_effect",
    "persona_summary",
    "socio_demographic_summary",
    "values_attitudes_perceptions_summary",
]

df_responses[audit_cols].sample(min(10, len(df_responses)), random_state=42)


,id_persona,segmento_e_bolhas,geração,ideologia,construct,answer_value,answer_label,fonte_familiar,rationale,source_familiarity_effect,persona_summary,socio_demographic_summary,values_attitudes_perceptions_summary
78,agent_5796645ef0d7,Conservadores cristãos,Geração Ordem e Progresso,Conservador(a),Família,3,Nem concordo nem discordo / moderado,False,A notícia sobre investimentos em tecnologia e ...,A falta de familiaridade com a fonte me leva a...,"Como conservador cristão, valorizo a família, ...",Minha condição de renda e ocupação me faz ver ...,"Embora a família seja central na minha vida, a..."
10,agent_2cc5d8ce623f,Progressistas,Geração Bossa Nova,Progressista,Confiança,4,Concordo parcialmente / alto,True,A notícia sobre investimento em tecnologia e s...,A familiaridade com a fonte aumenta minha conf...,"Como uma pessoa da Geração Bossa Nova, valoriz...",Minha renda e formação me permitem ver com oti...,Meu perfil progressista me leva a apoiar inova...
4,agent_2cc5d8ce623f,Progressistas,Geração Bossa Nova,Progressista,Ideologia,4,Concordo parcialmente / alto,True,Minha posição política progressista me faz val...,A familiaridade com a fonte me deixou mais rec...,"Como uma mulher progressista, valorizo iniciat...",Minha renda alta e formação me permitem ter um...,Meus valores progressistas influenciam minha i...
84,agent_5796645ef0d7,Conservadores cristãos,Geração Ordem e Progresso,Conservador(a),Comportamento_social,2,Discordo parcialmente / baixo,False,A notícia não se conecta diretamente com minha...,A falta de familiaridade com a fonte gera ceti...,"Como conservador cristão, valorizo mais a tran...",Minha situação de renda e ocupação me faz ques...,Minha visão conservadora me leva a ser cautelo...
64,agent_7e4fb86f81fc,Dependentes do Estado / Classe D e E,Geração.Com,Conservador(a),Insegurança,3,Nem concordo nem discordo / moderado,False,"A notícia traz um investimento positivo, mas t...",A falta de familiaridade com a fonte me faz in...,"Como um jovem de classe D e E, estou atento a ...",Minha condição de renda e ocupação me faz preo...,"Valorizo a transparência e a justiça social, e..."
68,agent_ccf4b0318027,Liberais sociais,Geração Ordem e Progresso,Muito progressista,Honestidade,4,Concordo parcialmente / alto,True,"A notícia apresenta um investimento positivo, ...",A familiaridade com a fonte aumenta minha conf...,"Como uma pessoa da Geração Ordem e Progresso, ...",Minha renda e educação me permitem ser crítico...,Minha visão progressista me leva a valorizar a...
30,agent_ac2595547fe5,Empresários,Geração Ordem e Progresso,Conservador(a),Identidade_nacional,4,Concordo parcialmente / alto,True,A notícia sobre investimento em tecnologia e s...,A familiaridade com a fonte me deixou mais rec...,"Como empresária, valorizo iniciativas que prom...",Minha posição como empresária e residente em u...,Meus valores conservadores e foco em estabilid...
45,agent_7cf1aaca19dd,Liberais sociais,Geração Redemocratização,Moderado(a),Família,4,Concordo parcialmente / alto,True,Vejo a importância da família em apoiar decisõ...,A familiaridade com a fonte me deixou mais rec...,"Como uma pessoa da Geração Redemocratização, v...",Minha situação de aposentada e a renda familia...,Meus valores liberais sociais me levam a ver a...
96,agent_9e8c2c2362b7,Agro,Geração Bossa Nova,Conservador(a),Identidade_nacional,3,Nem concordo nem discordo / moderado,False,A notícia sobre investimentos em tecnologia e ...,A falta de familiaridade com a fonte me faz ve...,Como uma mulher de 50-64 anos do segmento Agro...,Minha renda e ocupação me fazem valorizar inic...,Meu conservadorismo e ligação com a cultura ru...
11,agent_29ef94c4ffbf,Conservadores cristãos,Geração Redemocratização,Muito conservador(a),Religiosidade,4,Concordo parcialmente / alto,False,Minha fé influencia como vejo a responsabilida...,A falta de familiaridade com a fonte me faz se...,"Como conservadora cristã, valorizo a responsab...",Minha condição de funcionária pública e renda ...,Acred